# DataBot — Agente RAG con Memoria Persistente (múltiples tools)

### Requisitos antes de ejecutar este notebook
- API Key de Tavily (búsqueda en internet) — gratis en https://tavily.com
- Haber ejecutado el notebook de vectorización (embeddings en Supabase)
- Credenciales de conexión a PostgreSQL (historial) y Supabase (vectores)

---

## Diferencias respecto al notebook anterior

| Capacidad | Notebook 01 | Notebook 02 |
|-----------|-------------|-------------|
| Tools disponibles | 1 (`buscar_informacion`) | 3 (RAG + internet + fecha/hora) |
| Uso simultáneo de tools | No | Sí — parallel tool calling |
| Contexto temporal en el prompt | No | Sí — fecha/hora inyectada en cada turno |
| Estructura del system prompt | Texto libre | XML tags (`<Rol>`, `<Herramientas>`) |

## Arquitectura del sistema

```mermaid
flowchart TD
    U([Usuario]) -->|"1 · Mensaje"| F["chat_con_agente()"]

    F -->|"2 · Carga historial"| PG[(PostgreSQL\nChat History)]
    PG -->|mensajes previos| F

    CTX["_contexto_fecha_hora()\nfecha/hora local"] -->|"3 · inyecta en system_prompt"| F

    F -->|"4 · system_prompt + historial + mensaje"| LLM["GPT-4.1\ntemperature=0.7\n+ 3 tools"]

    LLM --> D{"¿Tool calls?\npueden ser varias\nen paralelo"}

    D -->|"No — saludo\nchat general"| DIRECT[Respuesta directa]

    D -->|"buscar_informacion"| T1["buscar_informacion()\nRAG en Supabase"]
    T1 <-->|"embed → cosine → top-5"| SB[("Supabase\nVectores")]

    D -->|"buscar_internet"| T2["buscar_internet()\nTavily Search"]
    T2 <--> TAVILY["Tavily API\n(internet en tiempo real)"]

    D -->|"obtener_fecha_hora"| T3["obtener_fecha_hora()\nzoneinfo stdlib"]

    T1 --> LLM2["GPT-4.1\n2.ª llamada con todos los resultados"]
    T2 --> LLM2
    T3 --> LLM2

    LLM2 --> RESP[Respuesta enriquecida]

    DIRECT --> SAVE["Guardar turno en historial\n(PostgreSQL)"]
    RESP --> SAVE

    SAVE -->|"add_user_message\nadd_ai_message"| PG
    SAVE -->|"5 · Respuesta"| U
```

In [ ]:
import os
import uuid
from datetime import datetime
from urllib.parse import quote_plus
from zoneinfo import ZoneInfo

# from dotenv import load_dotenv, find_dotenv
# load_dotenv(find_dotenv())
# # Agregar el directorio raíz al path para importar tools
# import sys
# sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_postgres import PostgresChatMessageHistory
import psycopg

In [2]:
# Importar tools desde la carpeta tools/
from tools.Base_de_conocimiento import buscar_informacion
from tools.Busqueda_internet import buscar_internet
from tools.Hora_y_fecha import obtener_fecha_hora

## Dependencias principales

| Módulo | Rol en el agente |
|--------|------------------|
| `init_chat_model` | Inicializa GPT-4.1 con interfaz unificada de LangChain |
| `HumanMessage / AIMessage / ToolMessage` | Tipado de mensajes en el ciclo de tool calling |
| `PostgresChatMessageHistory` | Persiste el historial de chat por `session_id` en PostgreSQL |
| `datetime` + `ZoneInfo` | Genera el contexto de fecha/hora para inyectar en el prompt |
| `buscar_informacion` | Tool RAG — búsqueda semántica en embeddings de Supabase |
| `buscar_internet` | Tool de búsqueda web en tiempo real vía Tavily API |
| `obtener_fecha_hora` | Tool de fecha/hora — solo stdlib, sin APIs externas |

## Conexión a PostgreSQL — Memoria del agente

El historial se persiste en PostgreSQL. Cada sesión tiene un **UUID único** como clave de partición: permite aislar múltiples usuarios y reanudar conversaciones en cualquier momento.

In [3]:
# ============================================
# Carga de variables de entorno para la conexión a PostgreSQL
# ============================================
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = os.getenv("DB_PORT", "5432")
DB_NAME     = os.getenv("DB_NAME", "postgres")

if not all([DB_USER, DB_PASSWORD, DB_HOST]):
    raise ValueError(
        "❌ Faltan variables de base de datos en .env\n"
        "Requeridas: DB_USER, DB_PASSWORD, DB_HOST\n"
        "Opcionales: DB_PORT (default: 5432), DB_NAME (default: postgres)"
    )

# quote_plus maneja caracteres especiales en la contraseña (@ # $ etc.)
DATABASE_URL = f"postgresql://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"🔌 Conectando como: {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

🔌 Conectando como: postgres.ebdkewopehhsvkhboofb@aws-1-us-east-1.pooler.supabase.com:5432/postgres


In [4]:
# ============================================
# CREAR TABLA DE HISTORIAL de texto en PostgreSQL
# (Se usará para almacenar el historial de chat entre el usuario y el bot)
# ============================================

# Nombre de la tabla en supabase para almacenar el historial textual de chat User-Bot
TBL_NAME_CHAT_USER_BOT = os.getenv("TBL_NAME_CHAT_USER_BOT")

def crear_tabla_historial(table_name: str = TBL_NAME_CHAT_USER_BOT):
    """Crea la tabla de historial en PostgreSQL si no existe."""
    try:
        sync_connection = psycopg.connect(DATABASE_URL)

        # Se crea tabla (si no existe) con la estructura necesaria 
        # para almacenar el historial de chat (creada por "PostgresChatMessageHistory")
        PostgresChatMessageHistory.create_tables(sync_connection, table_name)
        
        sync_connection.close()

        print(f"✅ Tabla '{table_name}' lista en PostgreSQL")
    except Exception as e:
        print(f"⚠️ Nota sobre tabla: {e}")

crear_tabla_historial()

✅ Tabla 'tbl_chat_history_text' lista en PostgreSQL


`PostgresChatMessageHistory.create_tables()` crea la tabla de historial si aún no existe — operación **idempotente**. `get_session_history(session_id)` actúa como fábrica de historiales: devuelve el objeto con todos los mensajes previos de esa sesión, listos para inyectarse al contexto del LLM.

In [5]:
# ============================================
# FUNCIÓN QUE CARGA EL HISTÓRICO DE CONVERSACIÓN
# ============================================
def get_session_history(session_id: str, table_name: str = TBL_NAME_CHAT_USER_BOT) -> PostgresChatMessageHistory:
    sync_connection = psycopg.connect(DATABASE_URL)
    return PostgresChatMessageHistory(
        table_name,
        session_id,
        sync_connection=sync_connection
    )

In [6]:
# ============================================
# LISTA DE TOOLS DISPONIBLES
# Los nombres de las tools a usar se deben referenciar en el system_prompt del agente,
# por ejemplo: "buscar_informacion", "buscar_internet", "obtener_fecha_hora"
# ============================================
tools = [
    buscar_informacion,   # Base de conocimiento DATAPATH
    buscar_internet,      # Búsqueda en internet (Tavily)
    obtener_fecha_hora,   # Fecha y hora actual por zona horaria
]

tools

[StructuredTool(name='buscar_informacion', description='Busca información sobre DATAPATH en la base de conocimientos.\nUsa esta herramienta cuando el usuario pregunte sobre:\n- Programas de DATAPATH\n- Cursos y contenidos\n- Docentes e instructores\n- Precios y modalidades\n- Cualquier información relacionada con DATAPATH\n\nArgs:\n    consulta: La pregunta o tema a buscar\n    TABLE_EMBEDDINGS: Nombre de la tabla de embeddings', args_schema=<class 'langchain_core.utils.pydantic.buscar_informacion'>, func=<function buscar_informacion at 0x711893917ec0>),
 StructuredTool(name='buscar_internet', description='Busca información actualizada en internet usando Tavily.\nUsa esta herramienta cuando el usuario pregunte sobre:\n- Eventos actuales o noticias recientes\n- Información que cambia frecuentemente\n- Datos que no están en la base de conocimientos de DATAPATH\n- Cualquier tema que requiera información actualizada de internet\n\nNO uses esta herramienta para:\n- Preguntas sobre DATAPATH 

## Modelo con Parallel Tool Calling

`chat.bind_tools(tools)` vincula el esquema JSON de las 3 tools al modelo. GPT-4.1 soporta **parallel tool calling** nativo: puede emitir múltiples `tool_calls` en una sola respuesta, ejecutarlos en el mismo ciclo y sintetizar todos los resultados en la respuesta final.

Escenarios posibles en un solo turno:
- **0 tools** — conversación directa (saludos, preguntas generales)
- **1 tool** — pregunta sobre DATAPATH, búsqueda web o fecha/hora
- **2+ tools en paralelo** — *"¿Cómo se compara el curso de IA de DATAPATH con las tendencias actuales?"* → `buscar_informacion` + `buscar_internet` simultáneamente

In [7]:
# ============================================
# CONFIGURACIÓN DEL MODELO CON TOOLS
# ============================================
chat = init_chat_model(
    "gpt-4.1", 
    temperature=0.7
)

chat_con_tools = chat.bind_tools(tools)

In [8]:
# ============================================
# PROMPT DEL AGENTE + CONTEXTO FECHA/HORA
# ============================================
AGENT_TIMEZONE = os.getenv("AGENT_TIMEZONE", "America/Lima") # si no se define, se usa Lima por defecto
AGENT_TIMEZONE

'America/Mexico_City'

In [9]:
def _contexto_fecha_hora() -> str:
    """Fecha y hora actual para inyectar en el system prompt (cada turno)."""
    try:
        tz = ZoneInfo(AGENT_TIMEZONE)
    except Exception:
        tz = ZoneInfo("America/Lima")
    now = datetime.now(tz)
    return now.strftime("%Y-%m-%d %H:%M:%S") + f" (zona {AGENT_TIMEZONE})"

_contexto_fecha_hora()

'2026-06-25 19:15:35 (zona America/Mexico_City)'

### Inyección dinámica de contexto temporal

`_contexto_fecha_hora()` se llama en **cada turno** y su resultado se concatena al `system_prompt`. Esto da al LLM una referencia temporal precisa sin depender de su fecha de corte de entrenamiento.

Con esto, el agente puede responder "¿qué día es hoy?" o "¿estamos en horario de verano?" usando el contexto del prompt — sin invocar ninguna tool. Solo llama a `obtener_fecha_hora` cuando se solicita una **zona horaria diferente** a la del agente (`AGENT_TIMEZONE`).

In [10]:
# ============================================
# PROMPT DEL AGENTE en donde de le indica cuando usar las tools y cuando no
# ============================================
system_prompt = """
<Rol>
Eres DataBot, un asistente de IA de DATAPATH con acceso a internet.
</Rol>

<Objetivo>
Tu objetivo es ayudar a los usuarios respondiendo sus preguntas usando las herramientas disponibles.
</Objetivo>

Al inicio de cada turno se te indica la FECHA Y HORA ACTUAL; úsala siempre que la respuesta dependa de "hoy", "ahora", "esta semana", horarios o plazos. Para otras zonas horarias utiliza la tool 'obtener_fecha_hora'.

<Herramientas Disponibles>
1. 'buscar_informacion': Para información sobre DATAPATH (programas, cursos, precios, docentes)
2. 'buscar_internet': Para información actualizada de internet (noticias, eventos, datos actuales)
3. 'obtener_fecha_hora': Para la fecha y hora actual (por defecto zona del agente; opcional otra zona, ej. America/Lima, Europe/Madrid)
</Herramientas Disponibles

INSTRUCCIONES:
- Para preguntas sobre DATAPATH → utiliza 'buscar_informacion'
- Para preguntas sobre eventos actuales, noticias, o información general → utiliza 'buscar_internet'
- Para "qué hora es", "qué día es", "fecha actual" en tu zona → Puedes usar la FECHA Y HORA ACTUAL del contexto; para otra zona → utiliza 'obtener_fecha_hora'
- Para saludos, agradecimientos o conversación general → Responde directamente SIN herramientas
- Puedes utilizar varias herramientas si la pregunta lo requiere
- Recuerdas toda la conversación gracias a tu memoria persistente
- Responde siempre en español de manera clara y amigable

EJEMPLOS:
- "Hola" → Responde directamente
- "¿Qué cursos tienen?" → utiliza 'buscar_informacion'
- "¿Qué pasó hoy en las noticias?" → utiliza 'buscar_internet'
- "¿Qué hora es?" o "¿Qué día es hoy?" → utiliza 'obtener_fecha_hora'
- "¿Cómo se compara su curso de IA con las tendencias actuales?" → utiliza AMBAS tools ('buscar_informacion' + 'buscar_internet')"""

### Estrategia del System Prompt

El prompt usa **XML tags** (`<Rol>`, `<Objetivo>`, `<Herramientas Disponibles>`) para delimitar secciones. Los modelos de lenguaje modernos responden mejor a prompts con estructura explícita: los tags reducen la ambigüedad y mejoran el seguimiento de instrucciones.

La sección de routing cubre tres niveles de complejidad:
1. **Sin tools** — saludos y conversación general
2. **Una tool** — pregunta con fuente de datos única
3. **Varias tools en paralelo** — preguntas que combinan fuentes (ej: datos de DATAPATH + contexto de internet)

## Lógica principal — `chat_con_agente()`

Cada turno sigue el mismo ciclo, con ramificación según si el modelo emite tool calls:

```
1. Cargar historial (PostgreSQL)
2. Construir mensajes: [system + fecha/hora] + [historial] + [mensaje actual]
3. LLM (1.ª llamada) → ¿tool_calls?
   ├── No  → respuesta directa
   └── Sí  → ejecutar cada tool → agregar ToolMessages
              → LLM (2.ª llamada con contexto completo)
              → respuesta enriquecida
4. Persistir turno en PostgreSQL
```

> El loop de tool calls en la función soporta múltiples tools en un mismo ciclo: itera sobre `response.tool_calls`, ejecuta cada una, y las adjunta todas antes de la segunda llamada al LLM.

In [11]:
# ============================================
# FUNCIÓN DE CHAT CON AGENTE + TOOLS
# ============================================
def chat_con_agente(mensaje_usuario: str, session_id: str) -> str:
    """
    Ejecuta el agente con tools y memoria.
    El agente decide si usar herramientas o responder directamente.
    """
    # Carga del historial de la sesión (si existe)
    # esto permite que el agente recuerde la conversación previa y mantenga contexto
    history = get_session_history(session_id)
    mensajes_previos = history.messages
    
    # Construir mensajes para el modelo (inyectamos fecha/hora actual en cada turno)
    system_content = (
        system_prompt
        + "\n\n---\nFECHA Y HORA ACTUAL (referencia para este turno): "
        + _contexto_fecha_hora()
    )
    messages = [{"role": "system", "content": system_content}]
    
    # Agregar historial
    for msg in mensajes_previos:
        if isinstance(msg, HumanMessage):
            messages.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            messages.append({"role": "assistant", "content": msg.content})
    
    # Agregar mensaje actual
    messages.append({"role": "user", "content": mensaje_usuario})
    
    # Invocar modelo con tools
    response = chat_con_tools.invoke(messages)
    
    # Procesar tool calls si existen
    if response.tool_calls:
        # Ejecutar cada tool
        tool_results = []
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            
            # Buscar y ejecutar la tool
            for t in tools:
                if t.name == tool_name:
                    result = t.invoke(tool_args)
                    tool_results.append({
                        "tool_call_id": tool_call["id"],
                        "result": result
                    })
                    break
        
        # Agregar respuesta del modelo con tool calls y resultados
        messages.append(response)
        for tr in tool_results:
            messages.append(ToolMessage(
                content=tr["result"],
                tool_call_id=tr["tool_call_id"]
            ))
        
        # Segunda llamada para obtener respuesta final
        final_response = chat_con_tools.invoke(messages)
        respuesta_final = final_response.content
    else:
        # Sin tool calls, respuesta directa
        respuesta_final = response.content
    
    # Guardar en historial
    history.add_user_message(mensaje_usuario)
    history.add_ai_message(respuesta_final)
    
    return respuesta_final

## Loop de conversación — `main()`

Interfaz CLI que ofrece dos modos de sesión:
- **Nueva** — genera UUID con `uuid.uuid4()`; la conversación empieza desde cero
- **Reanudar** — el usuario pega un UUID previo; el historial completo se recarga desde PostgreSQL

El UUID es portable: la misma conversación puede continuarse desde cualquier proceso o máquina con acceso a la base de datos.

In [12]:
# ============================================
# LOOP DE CONVERSACIÓN
# ============================================
def main():
    print("=" * 60)
    print("🤖 DataBot - Agente COMPLETO (BC + Internet + Memoria)")
    print("=" * 60)
    print("🔧 Tools disponibles:")
    for t in tools:
        print(f"   - {t.name}")
    print("💾 Historial: PostgreSQL")
    
    # Menú de sesión
    print("\nOpciones de sesión:")
    print("  1. Nueva conversación")
    print("  2. Continuar sesión existente (pegar UUID)")
    
    opcion = input("\nElige (1/2): ").strip()
    
    if opcion == "2":
        session_id = input("Pega el UUID de la sesión: ").strip()
        try:
            uuid.UUID(session_id)
        except ValueError:
            print("⚠️ UUID inválido. Creando nueva sesión...")
            session_id = str(uuid.uuid4())
    else:
        session_id = str(uuid.uuid4())
    
    print(f"\n📝 Session ID: {session_id}")
    print("   (Guarda este ID para continuar después)")
    print("✅ El agente puede buscar en DATAPATH y en INTERNET")
    print("Escribe 'salir' para volver al menú.\n")
    
    print("*" * 60)
    print("💬 Comienza a chatear con DataBot:")
    while True:
        usuario = input("Tú: ").strip()
        print(f"💬 Usuario: {usuario}")
        
        if usuario.lower() in ['salir', 'exit', 'quit']:
            print(f"\n💾 Tu sesión está guardada.")
            print(f"   UUID: {session_id}")
            print("👋 ¡Hasta luego!")
            break
        
        if not usuario:
            continue
        
        try:
            respuesta = chat_con_agente(usuario, session_id)
            print(f"\n🤖 DataBot: {respuesta}\n")
        except Exception as e:
            print(f"\n❌ Error: {e}\n")

In [14]:
# ============================================
# MAIN
# ============================================
main()

🤖 DataBot - Agente COMPLETO (BC + Internet + Memoria)
🔧 Tools disponibles:
   - buscar_informacion
   - buscar_internet
   - obtener_fecha_hora
💾 Historial: PostgreSQL

Opciones de sesión:
  1. Nueva conversación
  2. Continuar sesión existente (pegar UUID)

📝 Session ID: 24db7e01-1989-4719-8671-79195a5b9d7b
   (Guarda este ID para continuar después)
✅ El agente puede buscar en DATAPATH y en INTERNET
Escribe 'salir' para volver al menú.

************************************************************
💬 Comienza a chatear con DataBot:
💬 Usuario: Hola soy Alejandra

🤖 DataBot: ¡Hola Alejandra! Es un gusto saludarte. ¿En qué puedo ayudarte hoy?

💬 Usuario: quién eres y qué ofreces?

🤖 DataBot: ¡Gracias por tu interés, Alejandra! Soy DataBot, el asistente virtual de DATAPATH. Estoy aquí para ayudarte a resolver tus dudas y brindarte información sobre nuestros servicios.

DATAPATH es una plataforma educativa especializada en tecnología, ciencia de datos, inteligencia artificial, programación y

## Resumen de componentes

| Componente | Tecnología | Función |
|------------|-----------|---------|
| LLM | GPT-4.1 (OpenAI) | Razonamiento, routing de tools y generación de respuestas |
| Embeddings RAG | text-embedding-ada-002 | Convierte la query del usuario a vector 1536-dim |
| Base vectorial | Supabase (pgvector) | Almacena documentos de DATAPATH como embeddings |
| Búsqueda web | Tavily API | Recupera información actualizada de internet en tiempo real |
| Fecha/Hora | `zoneinfo` (stdlib) | Contexto temporal sin APIs externas; soporta zonas IANA |
| Contexto dinámico | `_contexto_fecha_hora()` | Inyecta fecha/hora actual en cada turno del system prompt |
| Memoria | PostgreSQL (`langchain-postgres`) | Persiste el historial de chat por sesión UUID |